# 07a — Scenario B Robustness Sweeps (Greedy)

This notebook evaluates how stable Scenario B portfolio selection is under:
- portfolio size `K`
- cost/safety weights (`lambda_cost`, `lambda_safety`)
- feature noise (`noise_sigma`) injected into scoring components

Method: fast greedy exact-K + swap-improve baseline.

Outputs:
- `data/results/07a_greedy_robustness_sweep_summary.csv`
- `data/results/07a_greedy_top_config_table.csv`
- `data/results/07a_greedy_top_configs_selected_trials.csv`


In [1]:
# ============================================================
# Cell 1 — Setup: imports, paths, and artifact checks
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

RESULTS_DIR = Path("data/results")
PROCESSED_DIR = Path("data/processed")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

PATH_CANDIDATES = PROCESSED_DIR / "scenario_B_candidates.csv"
PATH_CLASSICAL_SUMMARY  = RESULTS_DIR / "scenario_B_classical_summary.csv"
PATH_CLASSICAL_SELECTED = RESULTS_DIR / "scenario_B_classical_selected_trials.csv"

for p in [PATH_CANDIDATES, PATH_CLASSICAL_SUMMARY, PATH_CLASSICAL_SELECTED]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

print("OK: Found Scenario B artifacts.")
print("Candidates:", PATH_CANDIDATES)
print("Classical summary:", PATH_CLASSICAL_SUMMARY)
print("Classical selected:", PATH_CLASSICAL_SELECTED)


OK: Found Scenario B artifacts.
Candidates: data/processed/scenario_B_candidates.csv
Classical summary: data/results/scenario_B_classical_summary.csv
Classical selected: data/results/scenario_B_classical_selected_trials.csv


### What Cell 1 Just Did

- Established paths for Scenario B robustness sweeps.
- Verified the Scenario B candidates and classical-baseline artifacts exist before we run sweeps.


In [2]:
# ============================================================
# Cell 2 — Load candidates + normalize required components
# ============================================================

candidates = pd.read_csv(PATH_CANDIDATES)

required = ["nct_id", "_benefit_raw", "_cost_raw", "_safety_raw"]
missing = [c for c in required if c not in candidates.columns]
if missing:
    raise ValueError(f"Candidates missing required columns: {missing}\nFound: {list(candidates.columns)}")

candidates["nct_id"] = candidates["nct_id"].astype(str)
for col in ["_benefit_raw", "_cost_raw", "_safety_raw"]:
    candidates[col] = pd.to_numeric(candidates[col], errors="coerce").fillna(0.0).astype(float)

candidates = candidates.drop_duplicates(subset=["nct_id"]).reset_index(drop=True)

print("Candidates loaded:", candidates.shape)
display(candidates.head(8))


Candidates loaded: (60, 9)


,nct_id,brief_title,lead_sponsor,overall_status,phase,_benefit_raw,_cost_raw,_safety_raw,_pool_score
0,NCT06760637,Study of PF-07220060 With Letrozole in Adults ...,Pfizer,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
1,NCT06713616,PCORI Comparative Effectiveness Study-Esketami...,Yale University,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
2,NCT06711887,Phase III Extension Study of Efficacy and Safe...,Novartis Pharmaceuticals,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
3,NCT06706817,A Study to Investigate Changes in Symptoms in ...,AstraZeneca,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
4,NCT05674305,Radiotherapy Alone Versus Concurrent Chemo-rad...,Fudan University,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
5,NCT06703476,A Study of Surgical Techniques During Cystectomy,Memorial Sloan Kettering Cancer Center,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
6,NCT05675410,A Study to Compare Standard Therapy to Treat H...,National Cancer Institute (NCI),RECRUITING,PHASE3,1.0,2.0,0.0,0.744893
7,NCT06701331,Safety and Efficacy of Upadacitinib in Combina...,AbbVie,RECRUITING,PHASE3,1.0,2.0,0.0,0.744893


### What Cell 2 Just Did

- Loaded Scenario B candidates and enforced the required optimization features.
- Normalized numeric types for consistent downstream scoring and noise injection.


In [3]:
# ============================================================
# Cell 3 — Greedy solver + robustness sweep utilities
# ============================================================

from dataclasses import dataclass

def compute_weights(df, lambda_cost, lambda_safety):
    # minimize: -benefit + lambda_cost*cost + lambda_safety*safety
    return (-df["_benefit"].to_numpy(dtype=float)
            + lambda_cost * df["_cost"].to_numpy(dtype=float)
            + lambda_safety * df["_safety"].to_numpy(dtype=float))

def greedy_fixK_from_weights(df, weights, K):
    n = len(df)
    if K > n:
        raise ValueError(f"K={K} exceeds candidates={n}")
    idx = np.argsort(weights)[:K]
    x = np.zeros(n, dtype=int)
    x[idx] = 1

    best = float(np.sum(weights * x))
    improved = True
    while improved:
        improved = False
        ones = np.where(x == 1)[0]
        zeros = np.where(x == 0)[0]
        for i in ones:
            for j in zeros:
                x2 = x.copy()
                x2[i] = 0
                x2[j] = 1
                s2 = float(np.sum(weights * x2))
                if s2 < best:
                    x = x2
                    best = s2
                    improved = True
                    break
            if improved:
                break

    sel = np.where(x == 1)[0].tolist()
    return sel, best

def jaccard(a, b):
    a = set(a); b = set(b)
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)

def perturb_components(df, noise_sigma, seed):
    rng = np.random.default_rng(seed)
    out = df.copy()

    # Rename to standardized columns for this notebook
    out["_benefit"] = out["_benefit_raw"]
    out["_cost"] = out["_cost_raw"]
    out["_safety"] = out["_safety_raw"]

    if noise_sigma and noise_sigma > 0:
        for col in ["_benefit", "_cost", "_safety"]:
            base = out[col].to_numpy(dtype=float)
            scale = np.std(base) + 1e-9
            out[col] = base + rng.normal(0.0, noise_sigma * scale, size=len(out))

    return out

@dataclass(frozen=True)
class Config:
    K: int
    lambda_cost: float
    lambda_safety: float
    noise_sigma: float
    seed: int

print("OK: Greedy and robustness utilities ready.")


OK: Greedy and robustness utilities ready.


### What Cell 3 Just Did

- Implemented the Scenario B greedy exact-K + swap-improve selector.
- Added reproducible noise injection for robustness testing.
- Defined a `Config` object for clean sweep bookkeeping.


In [4]:
# ============================================================
# Cell 4 — Run robustness sweep + write artifacts
# ============================================================

# --- Sweep grid (start modest; expand later) ---
K_GRID = [6, 9, 12]
LAMBDA_COST_GRID = [0.5, 1.0, 2.0]
LAMBDA_SAFETY_GRID = [0.5, 1.0, 2.0]
NOISE_GRID = [0.0, 0.05, 0.10]
SEEDS = [0, 1, 2]

# Baseline reference (no noise, seed=0)
BASELINE = Config(K=9, lambda_cost=1.0, lambda_safety=1.0, noise_sigma=0.0, seed=0)

baseline_df = perturb_components(candidates, BASELINE.noise_sigma, BASELINE.seed)
baseline_w = compute_weights(baseline_df, BASELINE.lambda_cost, BASELINE.lambda_safety)
baseline_sel, baseline_score = greedy_fixK_from_weights(baseline_df, baseline_w, BASELINE.K)
baseline_set = set(baseline_df.iloc[baseline_sel]["nct_id"].tolist())

print("Baseline:", BASELINE)
print("Baseline score (linear-only):", baseline_score)
print("Baseline selected_n:", len(baseline_sel))

rows = []
sel_rows = []

total = len(K_GRID) * len(LAMBDA_COST_GRID) * len(LAMBDA_SAFETY_GRID) * len(NOISE_GRID) * len(SEEDS)
done = 0

for K in K_GRID:
    for lc in LAMBDA_COST_GRID:
        for ls in LAMBDA_SAFETY_GRID:
            for sigma in NOISE_GRID:
                # aggregate over seeds
                scores = []
                jac = []
                for seed in SEEDS:
                    cfg = Config(K=K, lambda_cost=lc, lambda_safety=ls, noise_sigma=sigma, seed=seed)
                    dfp = perturb_components(candidates, sigma, seed)
                    w = compute_weights(dfp, lc, ls)
                    sel, score = greedy_fixK_from_weights(dfp, w, K)
                    sel_ids = dfp.iloc[sel]["nct_id"].tolist()

                    scores.append(score)
                    jac.append(jaccard(sel_ids, baseline_set))

                    # store per-run selection (for top config export later)
                    sel_rows.append({
                        "K": K, "lambda_cost": lc, "lambda_safety": ls, "noise_sigma": sigma, "seed": seed,
                        "selected_n": len(sel_ids),
                        "score_linear_only": float(score),
                        "nct_ids": sel_ids
                    })

                    done += 1
                    if done % 25 == 0:
                        print(f"Progress: {done}/{total} configs complete")

                scores = np.array(scores, dtype=float)
                jac = np.array(jac, dtype=float)

                rows.append({
                    "K": K,
                    "lambda_cost": lc,
                    "lambda_safety": ls,
                    "noise_sigma": sigma,
                    "score_mean": float(np.mean(scores)),
                    "score_std": float(np.std(scores)),
                    "stability_jaccard_mean": float(np.mean(jac)),
                    "stability_jaccard_std": float(np.std(jac)),
                    "n_runs": int(len(SEEDS))
                })

sweep = pd.DataFrame(rows).sort_values(
    ["stability_jaccard_mean", "score_mean"],
    ascending=[False, True]
).reset_index(drop=True)

PATH_SUMMARY = RESULTS_DIR / "07a_greedy_robustness_sweep_summary.csv"
sweep.to_csv(PATH_SUMMARY, index=False)

# Top configs table
top_tbl = sweep.head(15).copy()
PATH_TOP_TABLE = RESULTS_DIR / "07a_greedy_top_config_table.csv"
top_tbl.to_csv(PATH_TOP_TABLE, index=False)

# Export selected trials for the top configs (seed=0 snapshot for readability)
sel_df = pd.DataFrame(sel_rows)
top_keys = set(tuple(r) for r in top_tbl[["K","lambda_cost","lambda_safety","noise_sigma"]].itertuples(index=False, name=None))
top_seed0 = sel_df[(sel_df["seed"] == 0) & (sel_df.apply(lambda r: (r["K"], r["lambda_cost"], r["lambda_safety"], r["noise_sigma"]) in top_keys, axis=1))].copy()

# explode nct_ids
top_seed0 = top_seed0.explode("nct_ids").rename(columns={"nct_ids": "nct_id"}).reset_index(drop=True)
PATH_TOP_SELECTED = RESULTS_DIR / "07a_greedy_top_configs_selected_trials.csv"
top_seed0.to_csv(PATH_TOP_SELECTED, index=False)

print("Wrote:")
print("  -", PATH_SUMMARY)
print("  -", PATH_TOP_TABLE)
print("  -", PATH_TOP_SELECTED)

display(sweep.head(10))


Baseline: Config(K=9, lambda_cost=1.0, lambda_safety=1.0, noise_sigma=0.0, seed=0)
Baseline score (linear-only): 9.0
Baseline selected_n: 9
Progress: 25/243 configs complete
Progress: 50/243 configs complete
Progress: 75/243 configs complete
Progress: 100/243 configs complete
Progress: 125/243 configs complete
Progress: 150/243 configs complete
Progress: 175/243 configs complete
Progress: 200/243 configs complete
Progress: 225/243 configs complete
Wrote:
  - data/results/07a_greedy_robustness_sweep_summary.csv
  - data/results/07a_greedy_top_config_table.csv
  - data/results/07a_greedy_top_configs_selected_trials.csv


,K,lambda_cost,lambda_safety,noise_sigma,score_mean,score_std,stability_jaccard_mean,stability_jaccard_std,n_runs
0,9,0.5,0.5,0.0,0.0,0.0,1.00,0.0,3
1,9,0.5,1.0,0.0,0.0,0.0,1.00,0.0,3
2,9,0.5,2.0,0.0,0.0,0.0,1.00,0.0,3
3,9,1.0,0.5,0.0,9.0,0.0,1.00,0.0,3
4,9,1.0,1.0,0.0,9.0,0.0,1.00,0.0,3
5,9,1.0,2.0,0.0,9.0,0.0,1.00,0.0,3
6,9,2.0,0.5,0.0,27.0,0.0,1.00,0.0,3
7,9,2.0,1.0,0.0,27.0,0.0,1.00,0.0,3
8,9,2.0,2.0,0.0,27.0,0.0,1.00,0.0,3
9,12,0.5,0.5,0.0,0.0,0.0,0.75,0.0,3


### What Cell 4 Just Did

- Ran a full robustness sweep over (K, λ_cost, λ_safety, noise, seed).
- Scored each configuration on:
  - mean linear objective score (lower is better under minimization),
  - and stability vs the baseline (Jaccard overlap).
- Wrote three artifacts:
  - full sweep summary,
  - top-config table,
  - and the selected trial IDs for top configs (seed=0 snapshot for auditability).


## Summary

Scenario B greedy robustness sweeps are complete.

- We now have a stability-ranked table of configurations across noise and weights.
- Next notebook: **07b QAOA robustness sweeps Scenario B**, using Aer if present (otherwise Aer-free statevector sampling), writing `07b_*` artifacts.
